# Non-contiguous bins for overlapping spectral templates

Two intensity components overlap on a one-dimensional spectrum. Their coefficients are the parameters of interest. Nearby wavelengths need not have similar coefficient sensitivity, so an information-aware bin can contain disconnected intervals.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import scorequant as fb
from examples.synthetic_problems import spectral_templates

problem = spectral_templates()
train, validation, test = problem.train, problem.validation, problem.test
train.observations.shape, train.scores.shape, problem.n_bins

## Explore observation space and score space

The observation has one coordinate, but each event has two score coordinates—one per coefficient. FisherBin imposes no ordering between these dimensions. The event weight represents the reference intensity.

In [ ]:
order = np.argsort(train.observations[:, 0])
x = train.observations[order, 0]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), constrained_layout=True)
axes[0].plot(x, train.weights[order])
axes[0].set(xlabel="spectral coordinate", ylabel="reference intensity", title="Observed intensity")
axes[1].plot(x, train.scores[order, 0], label="coefficient 1")
axes[1].plot(x, train.scores[order, 1], label="coefficient 2")
axes[1].set(xlabel="spectral coordinate", ylabel="score", title="Parameter sensitivity")
axes[1].legend();

## Compare the two library quantizers

Score k-means is the deterministic baseline. Soft Voronoi optimizes a differentiable information objective and then hardens the centers. Both return the same prediction and diagnostic contract.

In [ ]:
common = dict(
    weights=train.weights,
    n_bins=problem.n_bins,
    validation_scores=validation.scores,
    validation_weights=validation.weights,
)
kmeans = fb.fit_scores(train.scores, config=fb.KMeansConfig(seed=42, n_init=4), **common)
soft = fb.fit_scores(
    train.scores,
    config=fb.SoftVoronoiConfig(seed=42, n_init=3, max_steps=60, record_every=10),
    **common,
)
{
    "score k-means": kmeans.evaluate(test.scores, test.weights).geometric_mean_retention,
    "soft Voronoi": soft.evaluate(test.scores, test.weights).geometric_mean_retention,
}

In [ ]:
labels = np.asarray(soft.predict(test.scores))
order = np.argsort(test.observations[:, 0])
fig, ax = plt.subplots(figsize=(9, 3))
ax.scatter(test.observations[order, 0], labels[order], c=labels[order], cmap="tab20", s=5)
ax.set(
    xlabel="spectral coordinate",
    ylabel="hard-bin label",
    title="A label may reappear in disconnected spectral intervals",
);

## Interpretation

Disconnected intervals are not a defect: they have similar effects on the two coefficient estimates. If contiguous wavelength bands are a hard application constraint, this unconstrained score-space partition is the wrong tool unless that constraint is handled downstream.